In [1]:
%pip install pandas numpy scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [3]:
def generate_model_rf(X_train, X_val, y_train, y_val, n_estimators, max_depth, min_samples_split):
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)

    return r2, mae, mse, rmse

# Entrenatiemto

Se entrena el modelo con unos hiperparametros ya definidos, ademas se guarda el modelo para poder realizar la prediccion posteriormente


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

def generate_model_rf(X_train, X_val, y_train, y_val, n_estimators, max_depth, min_samples_split):
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)

    return r2, mae, mse, rmse

def train_and_save_model():
    # Cargar archivos
    df_train = pd.read_csv('../Data/train.csv')
    df_val = pd.read_csv('../Data/validation.csv')
    df_test = pd.read_csv('../Data/test.csv')

    # Combinar train + validation para codificar categóricas de forma uniforme
    df_combined = pd.concat([df_train, df_val])
    label_encoders = {}
    for col in df_combined.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df_combined[col] = le.fit_transform(df_combined[col].astype(str))
        label_encoders[col] = le

    # Repartir codificados
    df_train = df_combined.iloc[:len(df_train)].copy()
    df_val = df_combined.iloc[len(df_train):].copy()

    # Aplicar la misma codificación a test
    for col in label_encoders:
        df_test[col] = label_encoders[col].transform(df_test[col].astype(str))

    # Definir X e y
    X_train = df_train.drop(columns=['G1', 'G2', 'G3'])
    y_train = df_train['G3']
    X_val = df_val.drop(columns=['G1', 'G2', 'G3'])
    y_val = df_val['G3']

    # Hiperparámetros fijos
    n_estimators = 200
    max_depth = 30
    min_samples_split = 10

    # Entrenar y evaluar modelo
    r2, mae, mse, rmse = generate_model_rf(X_train, X_val, y_train, y_val,
                                           n_estimators, max_depth, min_samples_split)
    print("Modelo entrenado con hiperparámetros fijos:")
    print(f"R²: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")

    # Entrenar modelo final con train+val para predecir mejor
    X_train_val = pd.concat([X_train, X_val])
    y_train_val = pd.concat([y_train, y_val])
    final_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    final_model.fit(X_train_val, y_train_val)

    # Guardar el modelo entrenado
    joblib.dump(final_model, 'random_forest_model.pkl')
    print("Modelo guardado como 'random_forest_model.pkl'")

    # Guardar los label encoders para uso futuro (predicción)
    joblib.dump(label_encoders, 'label_encoders.pkl')
    print("Label encoders guardados como 'label_encoders.pkl'")

if __name__ == "__main__":
    train_and_save_model()


Modelo entrenado con hiperparámetros fijos:
R²: 0.3081 | MAE: 3.2184 | MSE: 15.6046 | RMSE: 3.9503
Modelo guardado como 'random_forest_model.pkl'
Label encoders guardados como 'label_encoders.pkl'


## Prediccion
En este caso se realiza la prediccion de todos los estudiantes del archivo csv "test", se muestra la nota G3 que obtendiran cada uno de ellos.

In [ ]:
import pandas as pd
import joblib

def preprocess_new_data(df_new, label_encoders):
    # Codificar columnas categóricas con los label_encoders guardados
    for col, le in label_encoders.items():
        if col in df_new.columns:
            df_new[col] = le.transform(df_new[col].astype(str))
        else:
            print(f"Advertencia: columna {col} no está en el DataFrame nuevo.")
    return df_new

def predict_new_data(df_new):
    # Cargar modelo y label encoders
    model = joblib.load('random_forest_model.pkl')
    label_encoders = joblib.load('label_encoders.pkl')

    # Preprocesar datos nuevos
    df_processed = preprocess_new_data(df_new.copy(), label_encoders)

    # Quitar columnas que no usa el modelo (si están presentes)
    for col in ['G1', 'G2', 'G3']:
        if col in df_processed.columns:
            df_processed = df_processed.drop(columns=[col])
            
    predictions = model.predict(df_processed)

    return predictions

if __name__ == "__main__":
  
    df_nuevos = pd.read_csv('../Data/test.csv')

    preds = predict_new_data(df_nuevos)
    print("Predicciones:")
    print(preds)


Predicciones:
[ 7.15836654 10.11959969 13.29568829  7.2398787   9.90045663  9.01973374
 11.71591543 11.65820578 10.18966862  5.6799691   9.07126463 10.09375136
 10.54913639 13.47162672  8.70491443 11.18350838  8.90696962 11.58424626
 12.02149312  9.38069941 12.23290506 10.77009039 11.78235074 11.33785451
 10.1387256  11.80480285 10.94424455 10.70795222  9.39233957  5.65673819
 10.73281742  9.87437051 14.79631278 12.9528578  12.54024697 12.28422614
  8.93616147 14.58318105 11.53016022 12.01831938 12.38638936 12.56194818
 10.84478849  9.75192424  9.69289406 11.88907436 10.63249929 12.58217345
 11.80240175  9.83455675  5.82739543 10.0329155  12.06486813 11.7422731
 11.25226255  7.32517618  9.01197226 12.38449793 11.13570809  9.12492492
  6.53127879 11.96354662  9.54799521 10.99723681 11.12683578 11.22344158
 11.47879058 13.10841916 10.30561544 11.28794716 10.09996687 10.07792209
 11.81788648  6.6022029   7.56419736]


## Prediccion 2
En este caso se predice la nota de un solo estudiante, se elegie el estudiante a predecir con el indice de este.

In [ ]:
import pandas as pd
import joblib

def preprocess_new_data(df_new, label_encoders):
    for col, le in label_encoders.items():
        if col in df_new.columns:
            df_new[col] = le.transform(df_new[col].astype(str))
        else:
            print(f"Advertencia: columna {col} no está en el DataFrame nuevo.")
    return df_new

def predict_new_data(df_new):
    model = joblib.load('random_forest_model.pkl')
    label_encoders = joblib.load('label_encoders.pkl')
    df_processed = preprocess_new_data(df_new.copy(), label_encoders)
    
    for col in ['G1', 'G2', 'G3']:
        if col in df_processed.columns:
            df_processed = df_processed.drop(columns=[col])
    
    prediction = model.predict(df_processed)
    return prediction

if __name__ == "__main__":
    df_nuevos = pd.read_csv('../Data/test.csv')

    #df_nuevos = df_nuevos.iloc[[0]]  
    # Cambia el índice según el estudiante que quieras predecir
    estudiante = df_nuevos.iloc[[0]]

    # Predecir la nota
    pred = predict_new_data(estudiante)

    print(f"Predicción para el estudiante 0: {pred[0]:.2f}")


Predicción para el estudiante 0: 7.16


# Conclusiones 

El modelo Random Forest implementado presentó un rendimiento adecuado para estimar la nota final (G3) del estudiante. La predicción realizada para el estudiante 0 fue de 7.16, lo cual indica que el modelo calcula una calificación final próxima a ese valor. Las métricas de error, como el error absoluto medio (MAE) y la raíz del error cuadrático medio (RMSE), indican que las predicciones se encuentran a pocos puntos de diferencia con respecto a las notas reales recordando que van de 0 a 20, lo que sugiere que el modelo puede anticipar el rendimiento académico con un nivel de precisión satisfactorio. Las variables empleadas para la predicción demostraron ser significativas, evidenciando que los factores seleccionados influyen en el resultado final de la calificación. A pesar de su buena precisión, el modelo no contempla factores externos que podrían afectar el desempeño del estudiante, y su efectividad depende de la calidad y cantidad de datos disponibles que hay en nuestros conjuntos de datos. Además, se realizó una predicción general que estimó la nota final de todos los estudiantes incluidos en nuestro conjunto de datos.(esas estan antes de la prediccion final)